In [3]:
pip install arxiv

In [21]:
pip install -U wikipedia

In [22]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper 

In [23]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1,doc_content_char_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [24]:
wiki.name

'wikipedia'

In [25]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")

docs = loader.load()

documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
).split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectordb = FAISS.from_documents(
    documents,
    embeddings
)

retriever = vectordb.as_retriever()

retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1193.87it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001D3516881C0>, search_kwargs={})

In [26]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith."
)

In [27]:
retriever_tool.name

'langsmith_search'

In [28]:
# arxiv tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_char_max=200)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv_tool.name

'arxiv'

In [29]:
tools = [wiki, retriever_tool, arxiv_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\stuf\\PROJECTS\\chatbot\\.venv310\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 StructuredTool(name='langsmith_search', description='Search for information about LangSmith.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001D34D122680>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x000001D34C55E9E0>),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000))]

In [30]:
pip install langchain_ollama

Note: you may need to restart the kernel to use updated packages.


In [31]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:4b"
)

In [32]:
pip install langsmith

In [33]:
import langchain
print(langchain.__version__)

1.3.4


In [34]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use tools when necessary."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

print(prompt)

input_variables=['input'] optional_variables=['agent_scratchpad'] input_types={'agent_scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Ann

In [35]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant."
)

In [36]:
pip install gemma

Note: you may need to restart the kernel to use updated packages.


In [42]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello"
            }
        ]
    }
)

print(response)

{'messages': [HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='139540f7-53a3-463d-8315-43c6ff0ef12b'), AIMessage(content="Hello! I'm here to help you with your questions. How can I assist you today? 😊", additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-11T10:43:23.3475125Z', 'done': True, 'done_reason': 'stop', 'total_duration': 196028462900, 'load_duration': 25705956900, 'prompt_eval_count': 319, 'prompt_eval_duration': 75399048000, 'eval_count': 139, 'eval_duration': 94421864000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019eb644-903e-78a3-ad3d-0b3f11857bb7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 319, 'output_tokens': 139, 'total_tokens': 458})]}


In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is langsmith?"
            }
        ]
    }
)   

print(response)

{'messages': [HumanMessage(content='What is langsmith?', additional_kwargs={}, response_metadata={}, id='c4c267c1-d5a4-4c77-b1cc-272f2355c4f4'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-06-11T11:52:42.2290758Z', 'done': True, 'done_reason': 'stop', 'total_duration': 428811076900, 'load_duration': 17620898600, 'prompt_eval_count': 323, 'prompt_eval_duration': 67198172999, 'eval_count': 647, 'eval_duration': 343843174000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--019eb680-789a-7a92-b084-04f2c3515a65-0', tool_calls=[{'name': 'langsmith_search', 'args': {'query': 'langsmith'}, 'id': '89fad9c9-fd73-4382-aceb-f003de413bea', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 323, 'output_tokens': 647, 'total_tokens': 970}), ToolMessage(content='Jennifer Wilby\nJohn N. Warfield\nKevin Warwick\nLudwig von Bertalanffy\nMaleyka Abbaszadeh\nManfred Clynes\nMargaret M

: 